# Tunda Box Junior Data Analyst Assessment — Solutions
This notebook reproduces key calculations, data-quality checks and statistical comparisons. It uses the case data supplied in the assessment preparation.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import proportions_ztest


## 1. Load the case data
The duplicate row for order 1006 is intentionally retained in the raw input so that the audit can identify it.

In [ ]:
customers = pd.DataFrame([
    (101, 'Nairobi', '2026-01-14', 'Organic'),
    (102, 'nairobi', '2026-02-03', 'Referral'),
    (103, 'Mombasa', '2026-06-20', 'Paid Social'),
    (104, 'Kisumu', '2026-03-02', 'Organic'),
    (105, 'Nairobi ', '2026-03-15', 'Paid Social'),
    (106, 'Nakuru', '2026-04-01', 'Referral'),
    (107, 'Mombasa', '2026-04-18', 'Organic'),
    (108, 'Nairobi', '2026-05-09', 'Paid Social'),
    (109, 'Eldoret', '2026-06-21', 'Organic'),
    (110, 'Kisumu', '2026-07-05', 'Paid Social'),
], columns=['customer_id','city','signup_date','channel'])
orders = pd.DataFrame([
    (1001,101,'2026-06-03','completed',2400,'M-Pesa'),
    (1002,102,'2026-06-05','completed',1850,'M-Pesa'),
    (1003,103,'2026-06-11','cancelled',3200,'Card'),
    (1004,104,'2026-06-18','completed',2100,'M-Pesa'),
    (1005,101,'2026-06-25','completed',2650,'Card'),
    (1006,105,'2026-07-02','completed',1500,'M-Pesa'),
    (1006,105,'2026-07-02','completed',1500,'M-Pesa'),
    (1007,106,'2026-07-09','refunded',2900,'Card'),
    (1008,102,'2026-07-14','completed',3100,'M-Pesa'),
    (1009,107,'2026-07-21','completed',None,'M-Pesa'),
    (1010,101,'2026-07-28','completed',2200,'M-Pesa'),
    (1011,108,'2026-08-04','completed',1750,'Card'),
    (1012,103,'2026-08-09','completed',2800,'M-Pesa'),
    (1013,104,'2026-08-15','completed',185000,'M-Pesa'),
    (1014,105,'2026-08-19','cancelled',1600,'M-Pesa'),
    (1015,102,'2026-08-27','completed',2450,'M-Pesa'),
    (1016,106,'2026-09-03','completed',3300,'Card'),
    (1017,101,'2026-09-08','completed',2750,'M-Pesa'),
    (1018,108,'2026-09-14','refunded',2000,'M-Pesa'),
    (1019,107,'2026-09-22','completed',2350,'Card'),
    (1020,999,'2026-09-26','completed',1900,'M-Pesa'),
], columns=['order_id','customer_id','order_date','status','amount_kes','payment_method'])
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
orders['order_date'] = pd.to_datetime(orders['order_date'])
display(customers.head(), orders.head())

## 2. Data-quality audit
Check duplicate order IDs, missing values, unusual amounts, unmatched customers, city formatting and orders dated before signup.

In [ ]:
duplicate_rows = orders[orders.duplicated(keep=False)].sort_values('order_id')
missing_amounts = orders[orders['amount_kes'].isna()]
outliers = orders[orders['amount_kes'] > 10000]
unmatched = orders.loc[~orders.customer_id.isin(customers.customer_id)]
city_variants = customers.assign(city_normalised=customers.city.str.strip().str.upper())
temporal_check = orders.merge(customers[['customer_id','signup_date']], on='customer_id', how='inner')
temporal_check = temporal_check[temporal_check.order_date < temporal_check.signup_date]
print('Exact duplicate rows:')
display(duplicate_rows)
print('Missing amounts:'); display(missing_amounts)
print('Unusually high amounts (screening threshold > KES 10,000):'); display(outliers)
print('Orders with unmatched customer IDs:'); display(unmatched)
print('City standardisation preview:'); display(city_variants[['city','city_normalised']].drop_duplicates())
print('Orders before signup:'); display(temporal_check)

## 3. Deduplicate exact repeated rows and analyse orders
For this sample, order 1006 is an exact duplicate. In production, conflicting rows sharing an order ID require an explicit canonical-record rule rather than arbitrary deletion.

In [ ]:
orders_clean = orders.drop_duplicates().copy()
status_counts = orders_clean.groupby('status')['order_id'].nunique().sort_index()
display(status_counts.rename('unique_orders').to_frame())
completed = orders_clean[orders_clean.status.eq('completed')].copy()
print('Unique order count:', orders_clean.order_id.nunique())
print('Completed order records:', completed.order_id.nunique())

## 4. Completed revenue by city and month
The city join excludes customer 999 from city attribution. NULL amounts are not treated as zero; pandas sum, like SQL SUM, skips missing values by default.

In [ ]:
city_revenue = (completed.merge(customers, on='customer_id', how='inner')
    .assign(city_clean=lambda d: d.city.str.strip().str.upper())
    .groupby('city_clean').amount_kes.sum().sort_values(ascending=False))
display(city_revenue.rename('revenue_kes').to_frame())
monthly_revenue = (completed.assign(month=completed.order_date.dt.to_period('M').astype(str))
    .groupby('month').amount_kes.sum().sort_index())
monthly = monthly_revenue.to_frame('revenue_kes')
monthly['previous_month'] = monthly.revenue_kes.shift(1)
monthly['pct_change'] = (monthly.revenue_kes / monthly.previous_month - 1) * 100
display(monthly)

## 5. Repeat purchase rate
Definition: at least two distinct completed orders. Denominator: all registered customers.

In [ ]:
completed_counts = completed.groupby('customer_id').order_id.nunique()
repeat_buyers = int((completed_counts >= 2).sum())
registered_customers = len(customers)
active_completed_buyers = int(completed_counts.index.nunique())
print(f'Repeat buyers: {repeat_buyers}')
print(f'All registered customers: {registered_customers}')
print(f'Repeat purchase rate (all customers): {repeat_buyers/registered_customers:.1%}')
print(f'Repeat purchase rate (completed buyers only): {repeat_buyers/active_completed_buyers:.1%}')

## 6. Mean and median order value
Use unique completed orders with known amounts. The median is more robust to the unusually high KES 185,000 transaction.

In [ ]:
known_amounts = completed.amount_kes.dropna()
print('Number of known completed amounts:', len(known_amounts))
print('Mean (KES):', known_amounts.mean())
print('Median including flagged outlier (KES):', known_amounts.median())
without_flagged = known_amounts[known_amounts != 185000]
print('Median excluding KES 185,000 for sensitivity only (KES):', without_flagged.median())

## 7. Conversion rates and traffic mix
Calculate overall and channel conversion rates. The channel-level comparison shows why an aggregate rate can move when the traffic mix changes.

In [ ]:
traffic = pd.DataFrame([
    ('September','Organic',40000,2400),
    ('September','Paid Social',10000,200),
    ('October','Organic',38000,2318),
    ('October','Paid Social',30000,630),
], columns=['month','channel','sessions','orders'])
traffic['conversion_rate_pct'] = traffic.orders / traffic.sessions * 100
display(traffic)
overall = traffic.groupby('month')[['sessions','orders']].sum()
overall['conversion_rate_pct'] = overall.orders / overall.sessions * 100
display(overall)
paid_share = traffic[traffic.channel.eq('Paid Social')].set_index('month').sessions / overall.sessions * 100
print('Paid Social share of sessions (%):')
display(paid_share)

## 8. A/B test: two-proportion comparison
Control: 50/1,000 = 5%. Variant: 60/1,000 = 6%. The observed relative uplift is 20%, but the two-sided test is not significant at alpha = 0.05.

In [ ]:
successes = np.array([50, 60])
samples = np.array([1000, 1000])
z_stat, p_value = proportions_ztest(successes, samples, alternative='two-sided')
control_rate = successes[0] / samples[0]
variant_rate = successes[1] / samples[1]
absolute_difference_pp = (variant_rate - control_rate) * 100
relative_lift_pct = (variant_rate / control_rate - 1) * 100
print(f'Control conversion: {control_rate:.1%}')
print(f'Variant conversion: {variant_rate:.1%}')
print(f'Absolute difference: {absolute_difference_pp:.1f} percentage point')
print(f'Relative uplift: {relative_lift_pct:.1f}%')
print(f'z statistic: {z_stat:.3f}; p-value: {p_value:.3f}')
print('Significant at 5%:', p_value < 0.05)

## 9. Conclusions and limitations
- Investigate the duplicate order, KES 185,000 outlier, missing amount, unmatched customer and date inconsistency against source systems.
- Treat the median as a more robust typical-order summary, but do not remove a transaction without verification.
- The aggregate conversion decline is largely a traffic-mix effect; assess channel profitability before changing spend.
- The A/B test's observed 20% relative uplift is not statistically significant with the supplied sample.
- This small case dataset supports an illustrative analysis, not definitive claims about long-term performance or causality.